In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
%pip install -q transformers datasets torch accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
from datasets import load_dataset

# 1. We load from the 'default' configuration in the converted parquet branch
# 2. We skip the 'all' name entirely as it is specific to the old .py script
print("Downloading HC3 Dataset (Verified Parquet Loading)...")
dataset = load_dataset(
    "Hello-SimpleAI/HC3", 
    "default", 
    revision="refs/convert/parquet", 
    split="train"
)

print("\nSUCCESS! Dataset loaded.")
print(f"Total rows: {len(dataset)}")
print(f"Sample data columns: {dataset.column_names}")

all/train/0000.parquet:   0%|          | 0.00/39.3M [00:00<?, ?B/s]

finance/train/0000.parquet:   0%|          | 0.00/5.18M [00:00<?, ?B/s]

medicine/train/0000.parquet:   0%|          | 0.00/1.40M [00:00<?, ?B/s]

open_qa/train/0000.parquet:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

reddit_eli5/train/0000.parquet:   0%|          | 0.00/30.3M [00:00<?, ?B/s]

wiki_csai/train/0000.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]


SUCCESS! Dataset loaded.
Total rows: 48644
Sample data columns: ['id', 'question', 'human_answers', 'chatgpt_answers', 'source']


In [4]:
import pandas as pd
from datasets import Dataset

print("Preprocessing data for RoBERTa training...")

# 1. Flatten the dataset: Convert list of answers into individual label rows
data_list = []
for item in dataset:
    # Add human answers as label 0 (Real)
    for text in item['human_answers']:
        if text and len(text.strip()) > 10: # Filter out empty or too-short text
            data_list.append({'text': text, 'label': 0})
            
    # Add ChatGPT answers as label 1 (Fake)
    for text in item['chatgpt_answers']:
        if text and len(text.strip()) > 10:
            data_list.append({'text': text, 'label': 1})

# 2. Convert to DataFrame for balancing
df = pd.DataFrame(data_list)

# 3. Create a balanced dataset (5,000 Human / 5,000 AI) for a high-precision demo
human_df = df[df['label'] == 0].sample(5000, random_state=42)
ai_df = df[df['label'] == 1].sample(5000, random_state=42)
balanced_df = pd.concat([human_df, ai_df]).sample(frac=1, random_state=42).reset_index(drop=True)

# 4. Convert back to Hugging Face format
final_dataset = Dataset.from_pandas(balanced_df)

print(f"Preprocessing Complete!")
print(f"Final training samples: {len(final_dataset)}")
print(f"Human samples: {len(human_df)} | AI samples: {len(ai_df)}")

Preprocessing data for RoBERTa training...
Preprocessing Complete!
Final training samples: 10000
Human samples: 5000 | AI samples: 5000


In [5]:
from transformers import AutoTokenizer

# --- SWAPPED TO ROBERTA ---
MODEL_NAME = "roberta-base"

print("Loading RoBERTa Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

print("Tokenizing the balanced dataset...")
tokenized_dataset = final_dataset.map(tokenize_function, batched=True)

# Split into Train and Test
full_ds = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_ds = full_ds["train"]
test_ds = full_ds["test"]

print(f"Tokenization Complete! Train size: {len(train_ds)}")

Loading RoBERTa Tokenizer...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing the balanced dataset...


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenization Complete! Train size: 8000


In [6]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# 1. Load RoBERTa with a classification head
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# 2. Define the metric (Accuracy)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 3. Training Arguments (Optimized for RoBERTa)
training_args = TrainingArguments(
    output_dir="./voxsentinel_text_engine",
    eval_strategy="epoch",        
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none",
    fp16=True, # RoBERTa handles this perfectly for fast training!
)

# 4. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

print("ROBERTA CONFIGURATION LOADED. Ready to train.")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ROBERTA CONFIGURATION LOADED. Ready to train.


In [7]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.064045,0.043989,0.991000
2,0.011528,0.025059,0.994500
3,0.000162,0.032414,0.995000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1500, training_loss=0.025244781355063122, metrics={'train_runtime': 1289.11, 'train_samples_per_second': 18.617, 'train_steps_per_second': 1.164, 'total_flos': 6314665328640000.0, 'train_loss': 0.025244781355063122, 'epoch': 3.0})